# AutoSpec AI — Exploration Notebook
### An Evidence-Grounded Automotive Specification Intelligence System

This notebook demonstrates the end-to-end AutoSpec AI pipeline on the 2014 Ford F-150 Service Manual:
1. **Document Intelligence**: Parsing and section detection
2. **Table-Aware Chunking**: Fastener torque tables and fluid capacities
3. **Hybrid Retrieval**: Combining dense FAISS vector search and sparse BM25Plus with RRF
4. **Evidence Grounding**: Source excerpt validation and cross-subsystem disambiguation

In [ ]:
import sys
from pathlib import Path

# Add src to Python path
sys.path.append(str(Path("..").resolve() / "src"))

from autospec.config import settings
from autospec.pipeline import AutoSpecPipeline
from autospec.citations import CitationFormatter
from autospec.export import ExportManager

print("AutoSpec AI environment ready!")

## 1. Initialize Pipeline & Ingest Sample Pages

In [ ]:
pipeline = AutoSpecPipeline(llm_provider="none")

# Reference manual path (pages 630 to 660 covering front & rear disc brakes)
manual_path = r"C:\Users\Dell\Downloads\sample-service-manual 1.pdf"

resp = pipeline.ingest_file(
    manual_path,
    start_page=630,
    end_page=660,
    force_rebuild=False,
)

print(f"Ingestion: {resp.status} | Chunks: {resp.chunk_count} | Cached: {resp.is_cached} | Time: {resp.processing_time_ms} ms")

## 2. Test Front vs Rear Brake Disambiguation

In [ ]:
# Query 1: Front brake guide pin bolts
q1 = "Torque for front brake caliper guide pin bolts"
res1 = pipeline.query(q1, top_k=1)
print("=== Query 1: Front Guide Pin ===")
for s in res1.results:
    print(f"[{s.verification_status.upper()}] {s.component}: {s.value} {s.unit} (Page {s.page_number}, {s.section})")

# Query 2: Rear brake guide pin bolts
q2 = "Torque for rear brake caliper guide pin bolts"
res2 = pipeline.query(q2, top_k=1)
print("\n=== Query 2: Rear Guide Pin ===")
for s in res2.results:
    print(f"[{s.verification_status.upper()}] {s.component}: {s.value} {s.unit} (Page {s.page_number}, {s.section})")

## 3. Formatted Citation & Grounding Inspection

In [ ]:
top_spec = res1.results[0]
print(CitationFormatter.format_card(top_spec))

## 4. Structured Export (CSV & JSON)

In [ ]:
all_specs = pipeline.get_all_specifications()
df = ExportManager.to_dataframe(all_specs)
print(f"Total indexed specifications: {len(df)}")
df.head(10)